# FinSight Intelligence | Company Research & Valuation

**Research prototype** · Fundamental financial snapshot · Dated price history · User-driven P/E sensitivities · Historical-return distributions · Linked recent company news

This notebook is **not** investment advice or a trained share-price forecaster. It deliberately distinguishes **P/E valuation assumptions** from **historical-price scenarios**. News classification is a separate research signal; it does not automatically influence the price equations.

**Install from repository root:** `python -m pip install -e '.[data,dev,intelligence,notebook]'`

**Data sources:** [yfinance](https://ranaroussi.github.io/yfinance/reference/api/yfinance.Ticker.html) (unofficial Yahoo interface), [GDELT DOC](https://blog.gdeltproject.org/gdelt-doc-2-0-api-debuts/) for article links, and [SEC EDGAR](https://www.sec.gov/edgar/search/) for primary-filings verification. Historical prices are not live quotes.

## 1 · Select company, target date and assumptions
These are **your assumptions**, not hidden price forecasts. Change the inputs and rerun the cells below.

In [ ]:
TICKER = "NVDA"                     # e.g. AAPL, MSFT, TSLA
HORIZON_MONTHS = 12                 # 1 to 60 months
NEWS_LOOKBACK_DAYS = 30             # 7, 30, 60, 90
EPS_GROWTH_PER_YEAR = 0.00          # 0% is illustrative, not evidence
PE_GROWTH_SPREAD = 0.10             # +/- 10 percentage points (bear vs bull)
TERMINAL_PE_SPREAD = 0.20           # +/- 20% on terminal multiple
SIMULATIONS = 3000


## 2 · Fetch and inspect company data
The notebook fails rather than generating fabricated data if market prices are missing. Some Yahoo summary fields may be unavailable, stale, or in a different reporting currency.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from datetime import date
from IPython.display import display
from finsight.intelligence.analytics import price_diagnostics, historical_distribution, rolling_interval_check
from finsight.intelligence.providers import fetch_company, fetch_company_news
from finsight.intelligence.valuation import earnings_valuation, current_multiple, horizon_date
from finsight.intelligence.sentiment import label_articles
from finsight.intelligence.report import build_report, report_json
from finsight.intelligence.quality import assess_quality

company = fetch_company(TICKER)
summary = price_diagnostics(company.closes)
quality = assess_quality(company, summary)
print("Data-quality gate:", "P/E valuation possible" if quality.valuation_ready else "P/E valuation blocked")
for issue in quality.issues:
    print("Data quality warning:", issue)
print(f"{company.company_name} ({company.ticker}) | quote currency: {company.currency or 'unknown'}")
print(f"Last adjusted price: {summary.last_adjusted_close:,.2f} as of {summary.end_date}")
print(f"Target date: {horizon_date(date.fromisoformat(summary.end_date), HORIZON_MONTHS)}")
print(f"Market data: {company.price_source}")


## 3 · Prices and financial metrics

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
company.closes.plot(ax=ax, lw=1.5, title=f"{company.ticker} — adjusted closing prices")
ax.set_ylabel(f"Adjusted close ({company.currency or 'currency unknown'})")
ax.grid(alpha=0.2)
plt.show()

selected = ["marketCap", "totalRevenue", "revenueGrowth", "earningsGrowth", "operatingMargins", "profitMargins", "freeCashflow", "totalCash", "totalDebt", "trailingEps", "trailingPE", "forwardEps"]
display(pd.DataFrame([{"Metric": field, "Source value": company.fundamentals.get(field)} for field in selected]))
print(f"Historical annualised realised volatility: {summary.annualised_volatility:.1%}")
print(f"Historical maximum drawdown: {summary.maximum_drawdown:.1%}")


## 4 · Historical-price distribution — NOT a calibrated forecast
Simulate future price outcomes by recombining historical returns in five-day blocks. This shows **what repeated historical return behaviour would imply**, not what the company *will* trade at.

In [ ]:
distribution = historical_distribution(company.closes, HORIZON_MONTHS, simulations=SIMULATIONS) if summary.observations >= 253 else None
walk_forward = rolling_interval_check(company.closes, HORIZON_MONTHS) if distribution is not None else None
if distribution is None:
    print("No distribution: fewer than 253 valid closes.")
else:
    display(pd.DataFrame([{"P10": distribution.p10, "Median": distribution.p50, "P90": distribution.p90, "Fraction below reference": distribution.fraction_below_start}]))
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.bar(["10th percentile", "Median", "90th percentile"], [distribution.p10, distribution.p50, distribution.p90])
    ax.set_ylabel(f"Price ({company.currency or 'unknown'})")
    ax.set_title("Empirical historical-return scenarios (not forecast confidence)")
    plt.show()
    print("Walk-forward diagnostic:", walk_forward)


## 5 · Transparent EPS × P/E scenario valuation
The reference market-implied P/E is based on trailing positive EPS. If earnings are negative or reporting currency differs, no P/E valuation is produced. Bear and bull cases vary the annual growth and terminal multiple; they are **not** statistically estimated confidence limits.

In [ ]:
trailing_eps = company.fundamentals.get("trailingEps")
reference_pe = current_multiple(summary.last_adjusted_close, trailing_eps)
currency_mismatch = bool(company.currency and company.financial_currency and company.currency != company.financial_currency)
TERMINAL_PE = reference_pe  # You can override this using a justified multiple, e.g. TERMINAL_PE = 20.0
valuations = None
if reference_pe is None or not quality.valuation_ready:
    print("P/E valuation withheld. Inspect the data-quality warnings above: earnings, date, currency and security type must be verified.")
else:
    valuations = [c.to_dict() for c in earnings_valuation(
        trailing_eps, summary.last_adjusted_close, HORIZON_MONTHS,
        EPS_GROWTH_PER_YEAR, TERMINAL_PE, PE_GROWTH_SPREAD, TERMINAL_PE_SPREAD,
    )]
    display(pd.DataFrame(valuations).rename(columns={"name": "Scenario", "implied_share_value": "Implied value", "change_from_reference": "Change vs current"}))
    print("All scenarios depend on user assumptions. They are not recommendations or consensus targets.")


## 6 · Recent sourced headlines (GDELT) and optional NLP labels
Search results are **not comprehensive or independently verified**. GDELT `seen_date` is a monitoring timestamp, not necessarily publication date. A trained local FinSight model (`artifacts/sentiment_model.joblib`) is optional. News sentiment is not used in share-price equations.

In [ ]:
news, news_search_url = [], None
try:
    news, news_search_url = fetch_company_news(company.company_name, NEWS_LOOKBACK_DAYS, limit=30)
except Exception as error:
    print(f"News unavailable: {error}")
news, sentiment_status = label_articles(news)
print(f"Headline NLP status: {sentiment_status}")
if news:
    display(pd.DataFrame(news)[["headline", "source", "seen_date", "url", "sentiment"]])
else:
    print("No linked recent headlines were returned; do not interpret this as absence of news.")


## 7 · Export the auditable research record
This JSON keeps the as-of date, raw source links, selected financial fields, user assumptions, available scenarios and important limitations.

In [ ]:
from pathlib import Path
warnings = ["Historical-return simulations are not calibrated future price forecasts.",
            "News sentiment is not a price signal and is excluded from valuation formulas.",
            "Yahoo summary fields may be stale or use mismatched reporting dates."]
warnings.extend(quality.issues)
if currency_mismatch:
    warnings.append("Quote and reporting currencies differ; P/E valuation withheld.")
valuation_inputs = {
    "reference_price": summary.last_adjusted_close,
    "price_as_of": summary.end_date,
    "trailing_eps": trailing_eps,
    "horizon_months": HORIZON_MONTHS,
    "eps_growth_assumption_annual": EPS_GROWTH_PER_YEAR,
    "terminal_pe": TERMINAL_PE,
    "bear_bull_eps_growth_spread_annual": PE_GROWTH_SPREAD,
    "bear_bull_terminal_pe_spread": TERMINAL_PE_SPREAD,
} if valuations is not None else None
record = build_report(company, summary.to_dict(), distribution.to_dict() if distribution else None, valuations, news, news_search_url, warnings, walk_forward, data_quality=quality.to_dict(), valuation_inputs=valuation_inputs)
out = Path(f"finsight_{company.ticker.lower()}_research.json")
out.write_text(report_json(record), encoding="utf-8")
print("Saved", out.resolve())


## Interpretation and research discipline

- **Never** represent the statistical price distribution as intrinsic value or a calibrated price forecast.
- **Never** claim the news model has outperformed markets: it currently classifies headline *language*.
- Verify financial metrics against the company's latest published annual report or SEC filing before acting.
- Compare the bear/base/bull assumptions with peers, earnings calls and industry evidence; input assumptions are your responsibility.
- Treat a missing model, unavailable market data, or an unavailable earnings multiple as a limitation—not a license to fill gaps with arbitrary numbers.
